#### Урок 9. Языковое моделирование.
**Задание.**

Разобраться с моделью генерации текста, собрать самим или взять датасет с вебинара и обучить генератор текстов.


In [1]:
# Подключаем диск
from google.colab import drive
drive.mount('./drive')

Drive already mounted at ./drive; to attempt to forcibly remount, call drive.mount("./drive", force_remount=True).


In [2]:
import tensorflow as tf

import numpy as np
import os
import time

In [3]:
path_to_file = '/content/drive/MyDrive/NLP/evgenyi_onegin.txt'

In [4]:
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# Delete Unicode spescial symbol \ufeff
text = text.replace('\ufeff', '')
# length of text is the number of characters in it
print('Length of text: {} characters'.format(len(text)))

Length of text: 286984 characters


In [5]:
print(text[:500])

Александр Сергеевич Пушкин

                                Евгений Онегин
                                Роман в стихах

                        Не мысля гордый свет забавить,
                        Вниманье дружбы возлюбя,
                        Хотел бы я тебе представить
                        Залог достойнее тебя,
                        Достойнее души прекрасной,
                        Святой исполненной мечты,
                        Поэзии живой и ясной,
                        Высо


In [6]:
text = text + text

In [7]:
# The unique characters in the file
vocab = sorted(set(text))
print('{} unique characters'.format(len(vocab)))

131 unique characters


In [8]:
# Creating a mapping from unique characters to indices
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

In [9]:
text_as_int, text[:2000], len(text_as_int), len(text)

(array([ 71, 110, 104, ..., 104, 121,   0]),
 'Александр Сергеевич Пушкин\n\n                                Евгений Онегин\n                                Роман в стихах\n\n                        Не мысля гордый свет забавить,\n                        Вниманье дружбы возлюбя,\n                        Хотел бы я тебе представить\n                        Залог достойнее тебя,\n                        Достойнее души прекрасной,\n                        Святой исполненной мечты,\n                        Поэзии живой и ясной,\n                        Высоких дум и простоты;\n                        Но так и быть - рукой пристрастной\n                        Прими собранье пестрых глав,\n                        Полусмешных, полупечальных,\n                        Простонародных, идеальных,\n                        Небрежный плод моих забав,\n                        Бессонниц, легких вдохновений,\n                        Незрелых и увядших лет,\n                        Ума холодных наблюде

In [10]:
# The maximum length sentence you want for a single input in characters
seq_length = 100
examples_per_epoch = len(text)//(seq_length+1)

# Create training examples / targets
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

for i in char_dataset.take(5):
    print(idx2char[i.numpy()])

А
л
е
к
с


In [11]:
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

for item in sequences.take(5):
    print(repr(''.join(idx2char[item.numpy()])))

'Александр Сергеевич Пушкин\n\n                                Евгений Онегин\n                          '
'      Роман в стихах\n\n                        Не мысля гордый свет забавить,\n                        '
'Вниманье дружбы возлюбя,\n                        Хотел бы я тебе представить\n                        '
'Залог достойнее тебя,\n                        Достойнее души прекрасной,\n                        Свят'
'ой исполненной мечты,\n                        Поэзии живой и ясной,\n                        Высоких д'


In [12]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

In [13]:
for input_example, target_example in  dataset.take(1):
    print('Input data: ', repr(''.join(idx2char[input_example.numpy()])))
    print('Target data:', repr(''.join(idx2char[target_example.numpy()])))

Input data:  'Александр Сергеевич Пушкин\n\n                                Евгений Онегин\n                         '
Target data: 'лександр Сергеевич Пушкин\n\n                                Евгений Онегин\n                          '


In [14]:
# Batch size
BATCH_SIZE = 64

BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

dataset

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [15]:
# Length of the vocabulary in chars
vocab_size = len(vocab)

# The embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

In [16]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim,
                                  batch_input_shape=[batch_size, None]),

        tf.keras.layers.LSTM(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),

        tf.keras.layers.LSTM(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),

         tf.keras.layers.LSTM(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),

        tf.keras.layers.LSTM(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),

        tf.keras.layers.Dense(vocab_size)
    ])
    return model

In [17]:
model = build_model(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    batch_size=BATCH_SIZE)

In [18]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, '# (batch_size, sequence_length, vocab_size)')

(64, 100, 131) # (batch_size, sequence_length, vocab_size)


In [19]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (64, None, 256)           33536     
                                                                 
 lstm (LSTM)                 (64, None, 1024)          5246976   
                                                                 
 lstm_1 (LSTM)               (64, None, 1024)          8392704   
                                                                 
 lstm_2 (LSTM)               (64, None, 1024)          8392704   
                                                                 
 lstm_3 (LSTM)               (64, None, 1024)          8392704   
                                                                 
 dense (Dense)               (64, None, 131)           134275    
                                                                 
Total params: 30592899 (116.70 MB)
Trainable params: 305

In [20]:
example_batch_predictions[0]

<tf.Tensor: shape=(100, 131), dtype=float32, numpy=
array([[-1.0933917e-05,  4.2909460e-06, -2.2703227e-05, ...,
         1.0726202e-05,  8.1841972e-06,  1.4494032e-05],
       [-3.0172567e-05,  3.0117477e-05, -5.9798236e-05, ...,
         3.3466880e-05,  1.0497095e-05,  3.5822843e-05],
       [-4.1767296e-05,  7.8587211e-05, -1.1306023e-04, ...,
         8.8718618e-05,  7.1821942e-06,  6.4748994e-05],
       ...,
       [ 3.4507469e-03, -2.2590600e-03, -5.6310766e-03, ...,
         1.4052963e-03, -1.3672560e-04,  3.6582197e-03],
       [ 3.4440896e-03, -1.9641020e-03, -5.4834657e-03, ...,
         1.5333794e-03, -2.4856208e-04,  3.2834280e-03],
       [ 3.4706162e-03, -1.6853644e-03, -5.3357123e-03, ...,
         1.6345913e-03, -3.3640536e-04,  2.9805761e-03]], dtype=float32)>

In [21]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()

In [22]:
print('Input: \n', repr(''.join(idx2char[input_example_batch[0]])))
print()
print('Next Char Predictions: \n', repr(''.join(idx2char[sampled_indices ])))

Input: 
 'о, почему;\n                        Да, впрочем, другу моему\n                        В том нужды было'

Next Char Predictions: 
 'WкOб4 щGwуM4AbЯfLЛяGЦ7!lк5чfЧ-Су1W19OГyзсzkcмGИИLШч\'Э2л(ЧgNеeс2яЬbtЬШЬzтб"N"УЖsТЯвБс7gэЮЧИCNd5E\'zeыс'


In [23]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

example_batch_loss = loss(target_example_batch, example_batch_predictions)
print('Prediction shape: ', example_batch_predictions.shape, ' # (batch_size, sequence_length, vocab_size)')
print('scalar_loss:      ', example_batch_loss.numpy().mean())

Prediction shape:  (64, 100, 131)  # (batch_size, sequence_length, vocab_size)
scalar_loss:       4.8763905


In [24]:
model.compile(optimizer='adam', loss=loss)

In [25]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt_{epoch}')

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_freq=7*10,
    save_weights_only=True)

In [27]:
%%time
EPOCHS = 100
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/100
88/88 [==============================] - 32s 358ms/step - loss: 2.1206
Epoch 2/100
88/88 [==============================] - 32s 353ms/step - loss: 1.6779
Epoch 3/100
88/88 [==============================] - 32s 358ms/step - loss: 1.4868
Epoch 4/100
88/88 [==============================] - 34s 375ms/step - loss: 1.4018
Epoch 5/100
88/88 [==============================] - 34s 367ms/step - loss: 1.3548
Epoch 6/100
88/88 [==============================] - 34s 371ms/step - loss: 1.3224
Epoch 7/100
88/88 [==============================] - 34s 371ms/step - loss: 1.3238
Epoch 8/100
88/88 [==============================] - 35s 389ms/step - loss: 1.2968
Epoch 9/100
88/88 [==============================] - 35s 376ms/step - loss: 1.2631
Epoch 10/100
88/88 [==============================] - 34s 376ms/step - loss: 1.2087
Epoch 11/100
88/88 [==============================] - 35s 378ms/step - loss: 1.1770
Epoch 12/100
88/88 [==============================] - 35s 391ms/step - loss: 1.1473
E

In [28]:
tf.train.latest_checkpoint(checkpoint_dir)

'./training_checkpoints/ckpt_100'

In [29]:
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)
model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
model.build(tf.TensorShape([1, None]))

In [30]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (1, None, 256)            33536     
                                                                 
 lstm_4 (LSTM)               (1, None, 1024)           5246976   
                                                                 
 lstm_5 (LSTM)               (1, None, 1024)           8392704   
                                                                 
 lstm_6 (LSTM)               (1, None, 1024)           8392704   
                                                                 
 lstm_7 (LSTM)               (1, None, 1024)           8392704   
                                                                 
 dense_1 (Dense)             (1, None, 131)            134275    
                                                                 
Total params: 30592899 (116.70 MB)
Trainable params: 3

In [31]:
def generate_text(model, start_string, temperature=1, num_generate=500):
    # Evaluation step (generating text using the learned model)

    # Number of characters to generate
    num_generate = num_generate

    # Converting our start string to numbers (vectorizing)
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    # Empty string to store our results
    text_generated = []

    # Low temperature results in more predictable text.
    # Higher temperature results in more surprising text.
    # Experiment to find the best setting.
    temperature = temperature

    # Here batch size == 1
    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)
        # using a categorical distribution to predict the character returned by the model
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # Pass the predicted character as the next input to the model
        # along with the previous hidden state
        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

Продемонстрируем, как обучилась сеть. Возьмем достаточно большую температуру (подобрана эмпирически):

In [35]:
text_ = generate_text(model, start_string=u'                       Мой дядя самых честных ', temperature=1.3)
print(text_)

                       Мой дядя самых честных преклостит
                Затянут, саковона!
                  XXXIV

                        В?                Я им ревеюще
                        Поклонник мирли и душа
         Ужин унылых узором
             XII

                        Так думал молодой повеса,
         ль охотно Апулея исли          Но здесь с победою поздравим
                        Конюшня, кухня и забор,
                        Я каждым утром пробужден
                         Видался с ним, и так нимало
            


In [36]:
text_ = generate_text(model, start_string=u'                       Мой дядя самых честных ', temperature=1)
print(text_)

                       Мой дядя самых честных правил,
                        Когда не в шутку занемог,
                        Он уважать себя заставил
                        И лучше велит приготовлился,
                                       Пониже подписи других
                        Они клевещут даже скучно;
                        В бесплодной сухости речей,
                        Расправил волоса рукой,
                        Вошел. Полна народу зала;
                                    Между жарким и блан-манже,
                


In [37]:
text_ = generate_text(model, start_string=u'                       Мой дядя самых честных ', temperature=0.01)
print(text_)

                       Мой дядя самых честных правил,
                        Когда не в шутку занемог,
                        Он уважать себя заставил
                        И лучше выдумать не мог.
                        Его пример другим наука;
                        Но, знать, сердечное страданье
                        Уже пришло ему невмочь.
                        Вот вам письмо его точь-в-точь.

                                                                                                                                       


In [38]:
!rm -rf ./training_checkpoints

In [39]:
!ls ./training_checkpoints

ls: cannot access './training_checkpoints': No such file or directory
